# InvokeAI on Google Colab — persistent setup (Google Drive)

Runs the latest InvokeAI release (web UI) on a Colab GPU, with all state stored on
Google Drive so nothing is lost when the runtime stops and you start it again.

**What persists on Drive** (`INVOKEAI_ROOT`): downloaded models, generated images
(`outputs`), the database (boards, gallery, workflows) and `invokeai.yaml` config.

**What is ephemeral**: the InvokeAI Python package itself — it is reinstalled into
Colab's fast local disk on every fresh runtime (a few minutes), keeping Drive I/O low.

## GPU
Use the most powerful GPU Google offers: **A100 (40 GB)** — it is the only Colab GPU
that fits the large modern models (Flux.1/Flux.2, SD 3.5 Large, Qwen Image) without
heavy offloading. Requires Colab Pro / Pro+. Fallback order: A100 > L4 (24 GB) > T4.
Set it in `Runtime -> Change runtime type -> A100 GPU`. The check cell warns if the
attached GPU is not an A100.

## How to use
1. `Runtime -> Change runtime type -> A100 GPU` before running anything.
2. Run the cells top to bottom. Cell 2 will ask you to authorize Google Drive.
3. The last cell prints a public `trycloudflare.com` URL — open it once the server
   has finished loading. Install models from the UI's **Model Manager**.

To stop: interrupt/stop the last cell. To resume later: rerun all cells — your
models and images are still on Drive.

In [ ]:
# Step 1: Verify GPU and Python before installing.
# InvokeAI requires Python >=3.11,<3.13. Recommended GPU: A100 (40 GB).
import sys
import subprocess

print(f"Python: {sys.version.split()[0]}")
if not (3, 11) <= sys.version_info[:2] <= (3, 12):
    print("WARNING: InvokeAI supports Python 3.11-3.12 only. Current runtime may fail to install.")

try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    print(f"GPU: {gpu}")
    if "A100" not in gpu:
        print("WARNING: A100 not detected. For large models (Flux, SD3.5, Qwen) select "
              "'A100 GPU' in Runtime -> Change runtime type (Colab Pro/Pro+).")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("ERROR: No NVIDIA GPU found. Set Runtime -> Change runtime type -> A100 GPU.")

In [ ]:
# Step 2: Mount Google Drive and point INVOKEAI_ROOT at it for persistence.
# Models, outputs, database and config live here, so they survive runtime restarts.
import os
from google.colab import drive

drive.mount("/content/drive")

INVOKEAI_ROOT = "/content/drive/MyDrive/invokeai"
os.makedirs(INVOKEAI_ROOT, exist_ok=True)

# Exported so both `invokeai-web` and the install step use the Drive location.
os.environ["INVOKEAI_ROOT"] = INVOKEAI_ROOT
os.environ["INVOKEAI_HOST"] = "0.0.0.0"
os.environ["INVOKEAI_PORT"] = "9090"

print(f"INVOKEAI_ROOT = {INVOKEAI_ROOT}")
print("Contents:", os.listdir(INVOKEAI_ROOT) or "(empty - first run)")

In [ ]:
# Step 3: Install the latest InvokeAI release with CUDA (cu128) wheels.
# The [cuda] extra pins a matching torch build; the package goes to Colab's local
# disk (fast), while INVOKEAI_ROOT on Drive holds the persistent data.
!pip install --upgrade "invokeai[cuda]" --extra-index-url https://download.pytorch.org/whl/cu128

import invokeai.version
print(f"InvokeAI installed: {invokeai.version.__version__}")

In [ ]:
# Step 4: Expose the server via a cloudflared tunnel and launch InvokeAI.
# cloudflared needs no signup; it prints a public https://*.trycloudflare.com URL.
# `invokeai-web` blocks this cell to keep the server running - stop the cell to shut down.
import subprocess
import re
import time

# Download cloudflared (idempotent).
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Start the tunnel to InvokeAI's port in the background.
log_path = "/content/cloudflared.log"
with open(log_path, "w") as log_file:
    subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{os.environ['INVOKEAI_PORT']}"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

# Poll the log until the public URL appears.
public_url = None
for _ in range(30):
    time.sleep(2)
    with open(log_path) as log_file:
        match = re.search(r"https://[-\w]+\.trycloudflare\.com", log_file.read())
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
print(f"InvokeAI public URL: {public_url or 'not ready - check /content/cloudflared.log'}")
print("Open it AFTER the server below reports it is running on port 9090.")
print("=" * 60)

# Launch the web server (foreground). Reads INVOKEAI_ROOT/HOST/PORT from the env.
!invokeai-web

## Notes

- **Persistence**: everything under `INVOKEAI_ROOT` (`/content/drive/MyDrive/invokeai`)
  is on Google Drive — models, `outputs`, the `databases/invokeai.db` and `invokeai.yaml`
  all survive a runtime stop. On the next session just rerun every cell; the package
  reinstalls (local, fast) but your data is read straight from Drive.
- **Models**: install them from the UI **Model Manager** (Starter Models / HuggingFace
  / URL). They download into the Drive root once and are reused forever after.
- **Drive space**: large models (Flux, SD3.5 Large, Qwen ~40 GB) consume a lot of Drive
  quota — make sure the account has enough free space.
- **SQLite on Drive**: the database lives on the Drive FUSE mount. For a single user this
  is fine; if you ever hit "database is locked", stop the server, wait a few seconds for
  Drive to sync, and restart. Avoid opening the same root from two runtimes at once.
- **Tunnel URL** changes every launch (quick tunnels are ephemeral). For a stable URL,
  swap cloudflared for an authenticated ngrok/cloudflare named tunnel.
- **GPU**: A100 (40 GB) gives the best throughput and fits the largest models; L4/T4 work
  for SDXL/SD1.5 but may need model offloading for Flux-class models.

## Shutdown

Run the cell below to stop cleanly after use. **First** stop the server: press the
**Stop** button on the Step 4 cell (or `Runtime -> Interrupt execution`) — the kernel
is busy while `invokeai-web` runs, so this cell can only execute once Step 4 is stopped.

It kills the tunnel, then flushes and unmounts Google Drive so all state is safely
written. To also free the GPU afterwards: `Runtime -> Disconnect and delete runtime`.

In [ ]:
# Step 5 (Shutdown): run AFTER interrupting the Step 4 server cell.
# Stops the tunnel/server and flushes + unmounts Drive so all state is persisted.
import subprocess

subprocess.run(["pkill", "-f", "cloudflared"], check=False)
subprocess.run(["pkill", "-f", "invokeai-web"], check=False)
print("Stopped cloudflared and invokeai-web (if still running).")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted - state is saved.")
except Exception as exc:
    print(f"Drive unmount skipped: {exc}")

print("Done. To free the GPU, use Runtime -> Disconnect and delete runtime.")